<a href="https://colab.research.google.com/github/virusds7778-coder/HSE_Denis_Ivanov_HOMEWORK_2026_2027/blob/main/lesson_1_%D0%94%D0%97_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: Юридический скрининг реестра контрагентов

Юротдел компании готовится к партнёрскому собранию. Перед ним — реестр контрагентов, с которыми компания планирует работу в следующем квартале: новые поставщики, заёмщики, инвестиционные партнёры.

До собрания нужно провести **юридический скрининг реестра**:

1. Каждого контрагента проверить по **чёрному списку Банка России** (компании с признаками нелегальной деятельности на финансовом рынке).
2. Для контрагентов с валютными договорами — определить **курс ЦБ на дату платежа** и зафиксировать расхождение «запрошенная дата / фактическая дата курса».
3. Присвоить каждому контрагенту **уровень риска** по правилам, которые вы сформулируете сами.
4. Подготовить **материалы для собрания**: Excel-дашборд и текстовое резюме с числами.

Реестр задан в ячейке ниже. Работайте с ним как с исходными данными: менять ИНН, суммы и даты нельзя, всё остальное — в ваших руках.

## Исходные данные

**Список контрагентов**. Поля: `name` (название), `inn` (ИНН, строка), `region` (субъект РФ), `contract_type` (тип договора), `amount` (сумма обязательства, руб.), `payment_date` (дата платежа по договору, `ДД.ММ.ГГГГ`).

⚠️ Данные реестра намеренно неоднородны: ИНН — строки, среди них встречаются записи с пробелами и лишними символами; суммы — целые числа; даты платежей — строки российского формата, часть дат попадает на выходные и праздники.

In [1]:
counterparties = [
    {"name": "ООО «ДРАЙВ ТРАСТ»",                 "inn": "7730311629",  "region": "Москва",                   "contract_type": "заём",       "amount": 4_500_000,  "payment_date": "10.03.2025"},
    {"name": "ООО «ЦПН КВ Групп»",                "inn": "7806622850",  "region": "Санкт-Петербург",          "contract_type": "инвестиции", "amount": 12_000_000, "payment_date": "15.04.2025"},
    {"name": "КПК «Центр финансового содействия»","inn": "1435279293",  "region": "Республика Саха (Якутия)", "contract_type": "заём",       "amount": 2_800_000,  "payment_date": "20.05.2025"},
    {"name": "ПК «ПОТОК»",                        "inn": "9204006374",  "region": "Республика Крым",          "contract_type": "инвестиции", "amount": 7_300_000,  "payment_date": "01.07.2025"},
    {"name": "ООО «УК Селигер 24»",               "inn": "6913017372",  "region": "Тверская область",         "contract_type": "инвестиции", "amount": 5_600_000,  "payment_date": "17.03.2025"},
    {"name": "ПК «Платформа Ру»",                 "inn": "7733473794",  "region": "Москва",                   "contract_type": "инвестиции", "amount": 9_900_000,  "payment_date": "23.02.2025"},
    {"name": "КПК «Сберкнижка»",                  "inn": "6452126313",  "region": "Саратовская область",      "contract_type": "заём",       "amount": 1_750_000,  "payment_date": "09.05.2025"},
    {"name": "СК «Образец»",                      "inn": "0571011790",  "region": "Чеченская Республика",     "contract_type": "страхование","amount": 3_250_000,  "payment_date": "16.06.2025"},
    {"name": "ООО «СтройМонтажСервис»",           "inn": " 7707083893", "region": "Москва",                   "contract_type": "подряд",     "amount": 9_500_000,  "payment_date": "12.04.2025"},
    {"name": "АО «ТранспортЛизинг»",              "inn": "5029066329",  "region": "Московская область",       "contract_type": "лизинг",     "amount": 6_200_000,  "payment_date": "28.04.2025"},
    {"name": "ООО «АгроТрейд Юг»",                "inn": "2310035747",  "region": "Краснодарский край",       "contract_type": "поставка",   "amount": 3_400_000,  "payment_date": "05.05.2025"},
    {"name": "ООО «ИТ-Интегратор»",               "inn": "7810142190",  "region": "Санкт-Петербург",          "contract_type": "услуги",     "amount": 1_900_000,  "payment_date": "19.05.2025"},
    {"name": "ООО «УралМетСнаб»",                 "inn": "6674287672",  "region": "Свердловская область",     "contract_type": "поставка",   "amount": 15_800_000, "payment_date": "26.04.2025"},
    {"name": "ООО «ВолгаСтрой»",                  "inn": "6315027260",  "region": "Самарская область",        "contract_type": "подряд",     "amount": 8_100_000,  "payment_date": "01.06.2025"},
    {"name": "ООО «НижегородОплет»",              "inn": "5260474345",  "region": "Нижегородская область",    "contract_type": "поставка",   "amount": 950_000,    "payment_date": "10.05.2025"},
    {"name": "ООО «СибЭнергоКомплект»",           "inn": "7709803000",  "region": "Москва",                   "contract_type": "услуги",     "amount": 4_100_000,  "payment_date": "22.06.2025"},
    {"name": "АО «БалтФлот Сервис»",              "inn": "7811018401",  "region": "Санкт-Петербург",          "contract_type": "услуги",     "amount": 2_450_000,  "payment_date": "14.01.2025"},
    {"name": "ООО «Средне-Волжский Завод»",       "inn": "6321049153",  "region": "Самарская область",        "contract_type": "поставка",   "amount": 11_300_000, "payment_date": "07.02.2025"},
    {"name": "ООО «СибирьАгроМаш»",               "inn": "5504001577",  "region": "Омская область",           "contract_type": "поставка",   "amount": 5_750_000,  "payment_date": "01.03.2025"},
    {"name": "ООО «КубаньАгроХим»",               "inn": "2366375063",  "region": "Краснодарский край",       "contract_type": "поставка",   "amount": 7_050_000,  "payment_date": "04.07.2025"},
    {"name": "ООО «КазаньСтройПроект»",           "inn": "1651007245",  "region": "Республика Татарстан",     "contract_type": "подряд",     "amount": 13_200_000, "payment_date": "15.08.2025"},
    {"name": "ООО «ВоронежТеплоСеть»",            "inn": "3664069397",  "region": "Воронежская область",      "contract_type": "услуги",     "amount": 2_150_000,  "payment_date": "07.10.2025"},
    {"name": "ООО «АлтайЭнергоПром»",             "inn": "2225006811",  "region": "Алтайский край",           "contract_type": "подряд",     "amount": 6_850_000,  "payment_date": "04.11.2025"},
    {"name": "ООО «Северный Порт»",               "inn": "2902098681",  "region": "Архангельская область",    "contract_type": "поставка",   "amount": 3_980_000,  "payment_date": "15.11.2025"},
    {"name": "ООО «ДальРыбФлот»",                 "inn": "2540166843",  "region": "Приморский край",          "contract_type": "поставка",   "amount": 18_400_000, "payment_date": "31.12.2025"},
    {"name": "ООО «РостовАгроСнаб»",              "inn": "6165094039",  "region": "Ростовская область",       "contract_type": "поставка",   "amount": 4_600_000,  "payment_date": "30.10.2025"},
    {"name": "ООО «ЕкатеринбургСигнал»",          "inn": "6671180619",  "region": "Свердловская область",     "contract_type": "услуги",     "amount": 1_380_000,  "payment_date": "21.11.2025"},
    {"name": "ООО «Жуковский АвиаТех»",           "inn": "5013025761",  "region": "Московская область",       "contract_type": "подряд",     "amount": 10_600_000, "payment_date": "12.06.2025"},
    {"name": "ООО «ЧелябинскМеталлТорг»",         "inn": "7444800868",  "region": "Челябинская область",      "contract_type": "поставка",   "amount": 8_900_000,  "payment_date": "01.09.2025"},
    {"name": "ООО «ЕкатеринбургСтройСнаб»",       "inn": "6673009275",  "region": "Свердловская область",     "contract_type": "подряд",     "amount": 5_300_000,  "payment_date": "30.10.2025"},
]

print(f"Контрагентов в реестре: {len(counterparties)}")

Контрагентов в реестре: 30


---
## Задание

Выполняйте блоки по порядку: выход каждого блока — вход следующего. Код пишите в ячейках с пометкой **«ВАШ КОД»**.

Требования ко всему коду:

- Все запросы к API — с `timeout` и `raise_for_status()`.
- Любой сетевой сбой не должен останавливать обработку реестра: ошибка фиксируется, обработка продолжается.
- Между запросами к API — пауза (`time.sleep`).
- Имена переменных и функций — на английском, комментарии — на русском.
- Код выполняется сверху вниз без ошибок.

---
## Блок 1. Скрининг по чёрному списку ЦБ

Проверьте **каждого** контрагента реестра по чёрному списку Банка России.

Что нужно сделать:

1. **Подготовить ИНН к проверке.** В реестре ИНН заданы строками, и не все они чистые: где-то лишний пробел, где-то другой «мусор». Приведите все ИНН к единому виду — строка из цифр без пробелов и посторонних символов. Добавьте в каждую запись поле `inn_clean`. Решите сами, как проверить корректность ИНН перед запросом. Некорректный ИНН — это не повод останавливать скрининг.
2. **Проверить каждый ИНН через API чёрного списка** (`Search` по фразе → при наличии кандидатов — `DetailInfo` → **точная сверка ИНН**). Помните: поиск по фразе находит подстроку — совпадение по фразе ещё не означает, что это та же организация.
3. **Зафиксировать результат проверки** для каждого контрагента: найден/не найден, а для найденных — название из реестра ЦБ, признаки, дата внесения, ликвидирована ли организация.
4. **Обеспечить устойчивость**: если запрос не удался (сеть, таймаут, ошибка сервера) — зафиксируйте сбой и продолжите проверку остальных. В конце выведите статистику: сколько проверено, сколько найдено в списке, сколько сбоев.

После блока у каждой записи реестра должны появиться поля с результатом проверки. Выведите итоговую таблицу: контрагент — статус проверки.

In [2]:
# ===================== БЛОК 1 — ВАШ КОД =====================
# 1. Очистка ИНН -> поле inn_clean
# 2. Проверка каждого ИНН через API (Search -> DetailInfo -> точная сверка)
# 3. Фиксация результата проверки в записи реестра
# 4. Устойчивость к сбоям + итоговая статистика

import re
import time
import requests

# Официальный сервис ЦБ РФ «Список компаний с выявленными признаками
# нелегальной деятельности на финансовом рынке» (Warning list API v1.0).
WB_API = "https://www.cbr.ru/warninglistapi"
REQUEST_TIMEOUT = 20          # таймаут одного запроса, сек
REQUEST_PAUSE = 0.4           # пауза между запросами к API

http = requests.Session()
http.headers.update({"User-Agent": "Mozilla/5.0 (legal-screening)"})


def clean_inn(raw_inn):
    """ИНН -> строка только из цифр: пробелы и «мусор» удаляются, ведущий ноль остаётся."""
    return re.sub(r"\D", "", str(raw_inn if raw_inn is not None else ""))


def is_valid_inn(inn):
    """Формальная корректность ИНН: ровно 10 цифр (юридическое лицо) или
    12 цифр (ИП, физическое лицо). Пустая строка и «мусор» не проходят."""
    return bool(re.fullmatch(r"\d{10}|\d{12}", inn))


def wb_search(phrase, page=0):
    """Search: поиск по фразе (название, адрес, сайт, ИНН). Возвращает кандидатов."""
    resp = http.get(WB_API + "/Search", params={"sphrase": phrase, "page": page},
                    timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return resp.json().get("Data") or []


def wb_detail(record_id):
    """DetailInfo: карточка организации из реестра ЦБ по id записи."""
    resp = http.get(WB_API + "/DetailInfo", params={"id": record_id}, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def to_ru_date(iso_date):
    """'2026-09-11T00:00:00' -> '11.09.2026'; пустое значение остаётся пустым."""
    if not iso_date:
        return None
    year, month, day = str(iso_date).split("T")[0].split("-")
    return day + "." + month + "." + year


def check_against_blacklist(inn):
    """Цепочка Search -> DetailInfo -> точная сверка ИНН. Сетевое исключение — на вызывающем."""
    result = {
        "cbr_status": "не найден в чёрном списке",
        "cbr_name": None,
        "cbr_signs": None,
        "cbr_added": None,
        "cbr_liquidated": None,
        "cbr_note": "",
    }

    candidates = wb_search(inn)
    time.sleep(REQUEST_PAUSE)
    if not candidates:
        return result

    for cand in candidates:
        # Поиск по фразе ищет подстроку: совпадение по фразе ещё не означает,
        # что найдена та же организация, поэтому сверяем ИНН из DetailInfo.
        try:
            details = wb_detail(cand["id"])
        except requests.RequestException as exc:
            result["cbr_note"] = "сбой DetailInfo: " + exc.__class__.__name__
            time.sleep(REQUEST_PAUSE)
            continue
        time.sleep(REQUEST_PAUSE)

        info = (details.get("Info") or [{}])[0]
        if clean_inn(info.get("inn")) != inn:
            continue

        result.update(
            cbr_status="в чёрном списке",
            cbr_name=info.get("nameOrg") or cand.get("nameOrg"),
            cbr_signs="; ".join(s.get("signRus", "") for s in details.get("Signs") or []
                                if s.get("signRus")) or None,
            cbr_added=to_ru_date(info.get("dt") or cand.get("dt")),
            cbr_liquidated=bool(info.get("isLikvid")),
        )
        break

    if result["cbr_status"] != "в чёрном списке" and not result["cbr_note"]:
        result["cbr_note"] = "по фразе найдено записей: " + str(len(candidates)) + ", ИНН совпадает: нет"
    return result


screen_stats = {"checked": 0, "found": 0, "not_found": 0, "bad_inn": 0, "failed": 0}

for record in counterparties:
    record["inn_clean"] = clean_inn(record["inn"])
    record["inn_valid"] = is_valid_inn(record["inn_clean"])
    screen_stats["checked"] += 1

    if not record["inn_valid"]:
        # Некорректный ИНН — не повод останавливать скрининг: фиксируем и идём дальше.
        record.update({"cbr_status": "ИНН некорректен — проверка невозможна",
                       "cbr_name": None, "cbr_signs": None, "cbr_added": None,
                       "cbr_liquidated": None,
                       "cbr_note": "после очистки: " + repr(record["inn_clean"])})
        screen_stats["bad_inn"] += 1
        continue

    try:
        record.update(check_against_blacklist(record["inn_clean"]))
    except requests.RequestException as exc:
        # Сеть, таймаут или 5xx: фиксируем сбой и продолжаем обработку реестра.
        record.update({"cbr_status": "сбой запроса к ЦБ", "cbr_name": None,
                       "cbr_signs": None, "cbr_added": None, "cbr_liquidated": None,
                       "cbr_note": exc.__class__.__name__})
        screen_stats["failed"] += 1
        continue

    if record["cbr_status"] == "в чёрном списке":
        screen_stats["found"] += 1
    else:
        screen_stats["not_found"] += 1

print("Итоги скрининга по чёрному списку ЦБ:")
print("  проверено контрагентов : " + str(screen_stats["checked"]))
print("  найдено в чёрном списке: " + str(screen_stats["found"]))
print("  не найдено             : " + str(screen_stats["not_found"]))
print("  некорректных ИНН       : " + str(screen_stats["bad_inn"]))
print("  сбоев сети             : " + str(screen_stats["failed"]))

print()
print("Найдены в чёрном списке ЦБ (данные реестра регулятора):")
for record in counterparties:
    if record["cbr_status"] == "в чёрном списке":
        state = "ликвидирована" if record["cbr_liquidated"] else "действует"
        print("  - " + record["name"] + " | ИНН " + record["inn_clean"])
        print("      из реестра ЦБ : " + str(record["cbr_name"]))
        print("      признаки      : " + str(record["cbr_signs"]))
        print("      дата внесения : " + str(record["cbr_added"]) + " | " + state)

print()
print("Статус проверки по всему реестру:")
print("Контрагент                        ИНН            Статус проверки ЦБ")
print("-" * 96)
for record in counterparties:
    print(record["name"][:30].ljust(34) + record["inn_clean"].ljust(15) + record["cbr_status"])


Итоги скрининга по чёрному списку ЦБ:
  проверено контрагентов : 30
  найдено в чёрном списке: 8
  не найдено             : 22
  некорректных ИНН       : 0
  сбоев сети             : 0

Найдены в чёрном списке ЦБ (данные реестра регулятора):
  - ООО «ДРАЙВ ТРАСТ» | ИНН 7730311629
      из реестра ЦБ : ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "ДРАЙВ ТРАСТ"
      признаки      : Признаки нелегального кредитора
      дата внесения : 11.09.2026 | действует
  - ООО «ЦПН КВ Групп» | ИНН 7806622850
      из реестра ЦБ : ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "ЦПН КВ ГРУПП" (БОНУСЛАЙН)
      признаки      : Признаки «финансовой пирамиды»
      дата внесения : 08.09.2026 | действует
  - КПК «Центр финансового содействия» | ИНН 1435279293
      из реестра ЦБ : КРЕДИТНЫЙ ПОТРЕБИТЕЛЬСКИЙ КООПЕРАТИВ "ЦЕНТР ФИНАНСОВОГО СОДЕЙСТВИЯ "АРГЫСКРЕДИТ"
      признаки      : Признаки «финансовой пирамиды»; Признаки нелегального кредитора
      дата внесения : 04.09.2026 | действует
  - ПК «ПОТОК» | ИНН 9204

---
## Блок 2. Курсовые данные

Для расчётов и отчёта нужен курс ЦБ на дату платежа каждого контрагента.

Что нужно сделать:

1. **Собрать уникальные даты платежей** реестра (без повторов) — и запрашивать курс один раз на дату, а не на каждую запись.
2. **Получить курс USD** на каждую дату через официальный источник ЦБ.
3. **Зафиксировать обе даты**: какую дату вы запросили и какую дату курс ЦБ **реально** вернул. Для части дат они не совпадут — это не ошибка, это свойство источника: курсы публикуются только по рабочим дням, и на выходной/праздник ЦБ возвращает курс последнего торгового дня. Оба значения должны попасть в результат.
4. **Сопоставить** каждому контрагенту курс его даты платежа (и фактическую дату курса).

После блока у каждой записи реестра должны появиться поля с курсом и фактической датой курса. Выведите таблицу «запрошенная дата → фактическая дата → курс USD» и отметьте, для каких дат курс сдвинулся.

In [3]:
# ===================== БЛОК 2 — ВАШ КОД =====================
# 1. Уникальные даты платежей реестра
# 2. Курс USD на каждую дату (официальный источник ЦБ)
# 3. Запрошенная дата + фактическая дата курса
# 4. Сопоставление курса каждому контрагенту

from datetime import datetime

# Официальный источник: XML-курсы ЦБ РФ. Ответ отдаётся в кодировке windows-1251.
RATES_URL = "https://www.cbr.ru/scripts/XML_daily.asp"
WEEKDAYS = ["понедельник", "вторник", "среда", "четверг", "пятница", "суббота", "воскресенье"]

# Уникальные даты: курс запрашивается один раз на дату, а не на каждую запись реестра.
unique_dates = sorted({datetime.strptime(record["payment_date"], "%d.%m.%Y").date()
                       for record in counterparties})
print("Уникальных дат платежей: " + str(len(unique_dates)) +
      " при количестве записей реестра: " + str(len(counterparties)) +
      " — запросов к ЦБ будет " + str(len(unique_dates)))


def fetch_usd_rate(day):
    """Курс USD ЦБ на запрошенную дату: возвращает (фактическая дата курса, курс)."""
    resp = http.get(RATES_URL, params={"date_req": day.strftime("%d.%m.%Y")},
                    timeout=REQUEST_TIMEOUT)
    resp.encoding = "windows-1251"          # XML ЦБ опубликован в windows-1251
    resp.raise_for_status()
    body = resp.text

    actual = re.search(r'<ValCurs Date="([^"]+)"', body)
    usd = re.search(r"<CharCode>USD</CharCode>.*?<VunitRate>([0-9,]+)</VunitRate>", body, re.S)
    if not actual or not usd:
        raise ValueError("в ответе ЦБ нет данных по USD за " + day.strftime("%d.%m.%Y"))

    # ЦБ пишет дробь запятой: 92,8457 -> 92.8457
    return actual.group(1), float(usd.group(1).replace(",", "."))


rates = {}      # запрошенная дата -> курс и фактическая дата курса
for day in unique_dates:
    requested = day.strftime("%d.%m.%Y")
    try:
        actual_date, usd_rate = fetch_usd_rate(day)
        rates[requested] = {"requested_date": requested,
                            "weekday": WEEKDAYS[day.weekday()],
                            "actual_date": actual_date,
                            "usd_rate": usd_rate,
                            "shifted": actual_date != requested,
                            "note": ""}
    except (requests.RequestException, ValueError) as exc:
        # Сбой не останавливает обработку реестра: фиксируем и идём дальше.
        rates[requested] = {"requested_date": requested, "weekday": WEEKDAYS[day.weekday()],
                            "actual_date": None, "usd_rate": None, "shifted": None,
                            "note": "сбой: " + exc.__class__.__name__}
    time.sleep(REQUEST_PAUSE)

# Сопоставление курса и фактической даты курса каждой записи реестра.
for record in counterparties:
    day_rates = rates[record["payment_date"]]
    record["usd_rate"] = day_rates["usd_rate"]
    record["rate_date"] = day_rates["actual_date"]
    record["rate_shifted"] = day_rates["shifted"]

print()
print("Запрошенная дата  День недели  Фактическая дата   Курс USD, руб  Что вернул ЦБ")
print("-" * 92)
shifted_days = 0
for day in unique_dates:
    data = rates[day.strftime("%d.%m.%Y")]
    if data["shifted"]:
        shifted_days += 1
    rate_text = "{:.4f}".format(data["usd_rate"]) if data["usd_rate"] is not None else "нет данных"
    comment = ("курс предыдущего рабочего дня" if data["shifted"]
               else data["note"] or "курс точно на запрошенную дату")
    print(data["requested_date"].ljust(18) + data["weekday"].ljust(14) +
          str(data["actual_date"]).ljust(18) + rate_text.ljust(15) + comment)

print()
print("Курс сдвинулся (платёж приходится на выходной/праздник) по датам: " + str(shifted_days))


Уникальных дат платежей: 29 при количестве записей реестра: 30 — запросов к ЦБ будет 29

Запрошенная дата  День недели  Фактическая дата   Курс USD, руб  Что вернул ЦБ
--------------------------------------------------------------------------------------------
14.01.2025        вторник       14.01.2025        102.7081       курс точно на запрошенную дату
07.02.2025        пятница       07.02.2025        96.9104        курс точно на запрошенную дату
23.02.2025        воскресенье   22.02.2025        88.1676        курс предыдущего рабочего дня
01.03.2025        суббота       01.03.2025        88.2568        курс точно на запрошенную дату
10.03.2025        понедельник   08.03.2025        89.1362        курс предыдущего рабочего дня
17.03.2025        понедельник   15.03.2025        85.5694        курс предыдущего рабочего дня
12.04.2025        суббота       12.04.2025        84.0040        курс точно на запрошенную дату
15.04.2025        вторник       15.04.2025        82.7671        курс 

---
## Блок 3. Правила риска

Присвойте каждому контрагенту **уровень риска**: `критический`, `высокий`, `средний`, `низкий`.

Систему правил вы разрабатываете **полностью сами** — это центральная часть задания.
**Что такое правило риска.** Правило — это условие на данных контрагента (и результатах Блоков 1–2), при котором сделка требует дополнительной проверки или отказа от неё. Хорошие правила опираются на юридическую логику: профиль мошенничества из чёрного списка, тип договора, масштаб обязательства, существенность контрагента для компании.

**Требования к системе:**

1. **Минимум четыре правила**, покрывающие все четыре уровня (`критический`, `высокий`, `средний`, `низкий`). Каждый уровень должен быть достижим хотя бы для одного контрагента реестра.
2. **Порядок старшинства**: если контрагенту подходят несколько правил, применяется самый высокий уровень. Порядок зафиксируйте в комментарии.
3. **Оформление каждого правила**: формулировка (комментарий перед кодом) → обоснование — почему это риск с точки зрения юриста (1–2 предложения) → реализация в коде.
4. **Осмысленность**: правила должны опираться на данные реестра и результаты Блоков 1–2. Правило, которое не зависит ни от чего и делит реестр на «нравится/не нравится», не засчитывается.

**Подсказки к направлениям** (не обязательные к использованию — это ориентиры, а не шаблон):

- сочетание типа договора с профилем контрагента (например, что означает для компании заём с организацией, у которой ЦБ выявил признаки нелегальной кредиторской деятельности?);
- масштаб обязательства (сумма) и как он влияет на приоритет проверки;
- состояние организации из чёрного списка (ликвидирована или действует) и что это меняет;
- сочетание нескольких факторов одновременно — правила-множители, повышающие уровень при совместном действии условий.

После блока у каждой записи реестра должно появиться поле с уровнем риска. Выведите распределение контрагентов по уровням и убедитесь, что все четыре уровня достижимы.

In [4]:
# ===================== БЛОК 3 — ВАШ КОД =====================
# Система правил риска: уровень присваивается первым сработавшим правилом,
# список ниже упорядочен от самого тяжёлого уровня к самому лёгкому.
#
# ПОРЯДОК СТАРШИНСТВА: П1 -> П2 -> П3 -> П4 -> П5 -> П6 -> П7 -> П8.
# Если контрагенту подходят несколько правил, применяется первое, то есть
# самый высокий уровень риска; младшие правила лишь поясняют картину.

LARGE_AMOUNT = 10_000_000     # обязательство, существенное для компании
MATERIAL_AMOUNT = 5_000_000   # обязательство, требующее усиленной проверки

# Сегменты финансового рынка: типы договоров компании и признаки нелегальной
# деятельности ЦБ, которые относятся к тому же сегменту.
SIGN_PROFILES = {
    "заём": ("нелегального кредитора", "финансовой пирамиды"),
    "инвестиции": ("финансовой пирамиды", "незаконного привлечения инвестиций",
                   "инвестиционной платформы", "рынка ценных бумаг"),
    "страхование": ("страховом рынке",),
    "лизинг": ("лизинг",),
}


def in_blacklist(rec):
    """Контрагент — фигурант чёрного списка ЦБ по точному совпадению ИНН."""
    return rec["cbr_status"] == "в чёрном списке"


def sign_matches_contract(rec):
    """Признак нелегальной деятельности ЦБ относится к тому же сегменту рынка,
    что и тип договора компании."""
    keywords = SIGN_PROFILES.get(rec["contract_type"], ())
    signs = rec["cbr_signs"] or ""
    return any(keyword in signs for keyword in keywords)


RISK_RULES = [
    # ПРАВИЛО 1 (критический). Контрагент в чёрном списке ЦБ, и признак нелегальной
    # деятельности относится к тому же сегменту финансового рынка, что и наш договор
    # (заём — нелегальное кредиторство, инвестиции — пирамиды и незаконный сбор средств,
    # страхование — нелегальное страхование).
    # Обоснование: компания по такому договору сама становится участником операций в сегменте,
    # где ЦБ уже зафиксировал нелегальную деятельность контрагента, — это риск блокировки
    # платежей по 115-ФЗ, оспаривания сделки и невозврата размещённых средств.
    ("критический", "П1. Чёрный список ЦБ + договор совпадающего финансового профиля",
     lambda r: in_blacklist(r) and sign_matches_contract(r)),

    # ПРАВИЛО 2 (критический). Фигурант чёрного списка ЦБ с обязательством от 10 млн руб.
    # либо уже ликвидированный по данным реестра ЦБ.
    # Обоснование: при таком масштабе убыток не покрывается резервами компании, а с
    # ликвидированным лицом обязательство не может быть исполнено в принципе — контрагента
    # как субъекта прав нет, взыскивать требование не с кого.
    ("критический", "П2. Чёрный список ЦБ + обязательство от 10 млн руб. или ликвидированная организация",
     lambda r: in_blacklist(r) and (r["amount"] >= LARGE_AMOUNT or bool(r["cbr_liquidated"]))),

    # ПРАВИЛО 3 (высокий). Контрагент в чёрном списке ЦБ, но договор хозяйственный
    # (поставка, подряд, услуги) либо признак ЦБ из другого сегмента рынка.
    # Обоснование: прямой финансовый риск ниже, но остаются налоговые претензии
    # (необоснованная налоговая выгода, «техническая» компания) и репутационный риск
    # работы с фигурантом реестра регулятора — сделка только после углублённого комплаенса.
    ("высокий", "П3. Чёрный список ЦБ при хозяйственном договоре",
     lambda r: in_blacklist(r)),

    # ПРАВИЛО 4 (высокий). Обязательство от 10 млн руб. при чистой проверке ЦБ.
    # Обоснование: сумма выше порога существенности — неисполнение заметно скажется на
    # финансовом результате, поэтому контрагент проверяется по полной процедуре
    # (платёжеспособность, обеспечение, полномочия подписанта).
    ("высокий", "П4. Существенное обязательство (от 10 млн руб.)",
     lambda r: r["amount"] >= LARGE_AMOUNT),

    # ПРАВИЛО 5 (высокий). Обязательство от 5 млн руб., дата платежа которого приходится
    # на день, когда ЦБ курс не устанавливал (результат Блока 2).
    # Обоснование: правило-множитель — крупная сумма и нерасчётный день платежа вместе
    # дают риск срыва срока: валюта обязательства считается по курсу предыдущего дня,
    # а перенос даты исполнения по ст. 193 и 314 ГК РФ сдвигает момент просрочки.
    ("высокий", "П5. Обязательство от 5 млн руб. + платёж в нерабочий для ЦБ день",
     lambda r: r["amount"] >= MATERIAL_AMOUNT and bool(r["rate_shifted"])),

    # ПРАВИЛО 6 (средний). Обязательство от 5 млн руб. без прочих отягчающих факторов.
    # Обоснование: сделка среднего масштаба не требует отказа, но стандартной сверки
    # реквизитов недостаточно — нужен проверочный лист юриста до подписания.
    ("средний", "П6. Обязательство от 5 млн руб.",
     lambda r: r["amount"] >= MATERIAL_AMOUNT),

    # ПРАВИЛО 7 (средний). Платёж в день, когда ЦБ курс не устанавливал (результат Блока 2).
    # Обоснование: дата платежа не совпадает с датой курса — риск ошибки в графике платежей
    # и спорной просрочки; график нужно сверить с производственным календарём.
    ("средний", "П7. Платёж в нерабочий для ЦБ день",
     lambda r: bool(r["rate_shifted"])),

    # ПРАВИЛО 8 (низкий). Фигурантом реестра ЦБ не является, обязательство до 5 млн руб.,
    # дата платежа — расчётный день.
    # Обоснование: совокупность факторов не выходит за пределы обычной хозяйственной
    # деятельности — достаточно стандартных условий договора и обычной сверки контрагента.
    ("низкий", "П8. Флагов нет: сумма до 5 млн руб., платёж в расчётный день",
     lambda r: True),
]

RISK_ORDER = ["критический", "высокий", "средний", "низкий"]


def assess_risk(rec):
    """Первое сработавшее правило определяет уровень риска контрагента."""
    for level, title, rule in RISK_RULES:
        if rule(rec):
            return level, title
    return RISK_RULES[-1][0], RISK_RULES[-1][1]


for record in counterparties:
    record["risk_level"], record["risk_rule"] = assess_risk(record)

print("Распределение контрагентов по уровням риска:")
print("Уровень       Кол-во   Сумма обязательств, руб")
print("-" * 58)
for level in RISK_ORDER:
    group = [r for r in counterparties if r["risk_level"] == level]
    total = sum(r["amount"] for r in group)
    print(level.ljust(14) + str(len(group)).ljust(9) + "{:,}".format(total).replace(",", " "))

print()
print("Какие правила сработали:")
for rule_level, title, _ in RISK_RULES:
    hits = [r for r in counterparties if r["risk_rule"] == title]
    print("  [" + rule_level + "] " + title + " — " + str(len(hits)) + " шт.")

print()
print("Все четыре уровня достигнуты на реестре: " +
      str(set(RISK_ORDER) == {r["risk_level"] for r in counterparties}))

print()
print("Реестр с уровнями риска:")
for record in counterparties:
    print("  [" + record["risk_level"] + "] " + record["name"] + " — " +
          record["contract_type"] + ", " + "{:,}".format(record["amount"]).replace(",", " ") +
          " руб., платёж " + record["payment_date"] + ", " + record["cbr_status"])


Распределение контрагентов по уровням риска:
Уровень       Кол-во   Сумма обязательств, руб
----------------------------------------------------------
критический   8        47 100 000
высокий       9        99 350 000
средний       8        37 950 000
низкий        5        14 560 000

Какие правила сработали:
  [критический] П1. Чёрный список ЦБ + договор совпадающего финансового профиля — 8 шт.
  [критический] П2. Чёрный список ЦБ + обязательство от 10 млн руб. или ликвидированная организация — 0 шт.
  [высокий] П3. Чёрный список ЦБ при хозяйственном договоре — 0 шт.
  [высокий] П4. Существенное обязательство (от 10 млн руб.) — 5 шт.
  [высокий] П5. Обязательство от 5 млн руб. + платёж в нерабочий для ЦБ день — 4 шт.
  [средний] П6. Обязательство от 5 млн руб. — 4 шт.
  [средний] П7. Платёж в нерабочий для ЦБ день — 4 шт.
  [низкий] П8. Флагов нет: сумма до 5 млн руб., платёж в расчётный день — 5 шт.

Все четыре уровня достигнуты на реестре: True

Реестр с уровнями риска:
  [критиче

---
## Блок 4. Материалы для собрания

Подготовьте два артефакта.

**4.1. Excel-дашборд** `screening_dashboard.xlsx` (библиотека openpyxl), три листа:

- **«Сводка»** — ключевые числа: всего контрагентов; сколько в чёрном списке; распределение по уровням риска; суммарная сумма обязательств по каждому уровню риска. Числа должны сходиться с реестром.
- **«Реестр»** — все 12 контрагентов с колонками: название, ИНН, регион, тип договора, сумма, дата платежа, статус проверки ЦБ, уровень риска, курс на дату платежа. Строки с риском `критический` — красная заливка, `высокий` — жёлтая, остальные — без заливки. Шапка — жирным, ширина колонок подобрана так, чтобы названия читались.
- **«Курсы»** — таблица «запрошенная дата → фактическая дата → курс USD» по всем уникальным датам.

**4.2. Текстовое резюме** `screening_summary.txt` (5–8 строк, с числами):

- сколько контрагентов проверено;
- сколько обнаружено в чёрном списке (и сколько из них ликвидировано);
- сколько контрагентов каждого уровня риска;
- суммарная экспозиция по критическому и высокому риску (в рублях);
- сколько дат платежей пришлось на выходные/праздники (курс сдвинулся).

Резюме должно читаться как вывод для партнёров: короткие утверждения с числами, без технических деталей.

In [5]:
# ===================== БЛОК 4 — ВАШ КОД =====================
# 4.1. Excel-дашборд: листы «Сводка», «Реестр», «Курсы»
# 4.2. Текстовое резюме screening_summary.txt

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

DASHBOARD_FILE = "screening_dashboard.xlsx"
SUMMARY_FILE = "screening_summary.txt"

BOLD = Font(bold=True)
RED_FILL = PatternFill("solid", fgColor="FFC7CE")       # критический риск
YELLOW_FILL = PatternFill("solid", fgColor="FFEB9C")    # высокий риск
MONEY_FORMAT = "#,##0"
RATE_FORMAT = "0.0000"

# --- Ключевые числа для сводки -------------------------------------------------
total_amount = sum(r["amount"] for r in counterparties)
blacklisted = [r for r in counterparties if r["cbr_status"] == "в чёрном списке"]
liquidated = [r for r in blacklisted if r["cbr_liquidated"]]
shifted_dates = [d for d in rates.values() if d["shifted"]]
level_stats = {level: {
    "count": len([r for r in counterparties if r["risk_level"] == level]),
    "amount": sum(r["amount"] for r in counterparties if r["risk_level"] == level),
} for level in RISK_ORDER}

# Числа сводки должны сходиться с реестром.
assert sum(s["count"] for s in level_stats.values()) == len(counterparties)
assert sum(s["amount"] for s in level_stats.values()) == total_amount

wb = Workbook()

# --- Лист «Сводка» -------------------------------------------------------------
sheet = wb.active
sheet.title = "Сводка"
sheet.append(["Показатель", "Значение"])
summary_rows = [
    ("Всего контрагентов в реестре", len(counterparties)),
    ("Проверено по чёрному списку ЦБ", screen_stats["checked"]),
    ("Найдено в чёрном списке ЦБ", len(blacklisted)),
    ("Из них ликвидировано", len(liquidated)),
    ("Не найдено в чёрном списке", screen_stats["not_found"]),
    ("Сбоев при проверке", screen_stats["failed"]),
    ("Сумма обязательств по реестру, руб.", total_amount),
]
for level in RISK_ORDER:
    summary_rows.append(("Контрагентов с риском «" + level + "»", level_stats[level]["count"]))
    summary_rows.append(("Сумма обязательств «" + level + "», руб.", level_stats[level]["amount"]))
summary_rows += [
    ("Уникальных дат платежей", len(rates)),
    ("Дат платежа сдвинутых (курс предыдущего дня)", len(shifted_dates)),
]
for row in summary_rows:
    sheet.append(list(row))
for cell in sheet[1]:
    cell.font = BOLD
for row_index in range(2, sheet.max_row + 1):
    if "руб" in str(sheet.cell(row=row_index, column=1).value):
        sheet.cell(row=row_index, column=2).number_format = MONEY_FORMAT
sheet.column_dimensions["A"].width = 46
sheet.column_dimensions["B"].width = 20

# --- Лист «Реестр» -------------------------------------------------------------
registry = wb.create_sheet("Реестр")
headers = ["Название", "ИНН", "Регион", "Тип договора", "Сумма, руб.", "Дата платежа",
           "Статус проверки ЦБ", "Уровень риска", "Курс USD на дату платежа",
           "Фактическая дата курса"]
registry.append(headers)
for record in counterparties:
    registry.append([
        record["name"],
        record["inn_clean"],
        record["region"],
        record["contract_type"],
        record["amount"],
        record["payment_date"],
        record["cbr_status"],
        record["risk_level"],
        record["usd_rate"],
        record["rate_date"],
    ])

for cell in registry[1]:
    cell.font = BOLD
    cell.alignment = Alignment(horizontal="center", wrap_text=True)

for row_index in range(2, registry.max_row + 1):
    level = registry.cell(row=row_index, column=8).value
    fill = RED_FILL if level == "критический" else (YELLOW_FILL if level == "высокий" else None)
    for column_index in range(1, len(headers) + 1):
        cell = registry.cell(row=row_index, column=column_index)
        if fill is not None:
            cell.fill = fill
    registry.cell(row=row_index, column=5).number_format = MONEY_FORMAT
    registry.cell(row=row_index, column=9).number_format = RATE_FORMAT
    registry.cell(row=row_index, column=2).alignment = Alignment(horizontal="center")

# Ширины колонок по самому длинному значению, чтобы названия читались.
for column_index in range(1, len(headers) + 1):
    letter = get_column_letter(column_index)
    longest = max(len(str(registry.cell(row=row, column=column_index).value or ""))
                  for row in range(1, registry.max_row + 1))
    registry.column_dimensions[letter].width = min(max(longest + 2, 12), 42)
registry.freeze_panes = "A2"

# --- Лист «Курсы» --------------------------------------------------------------
rates_sheet = wb.create_sheet("Курсы")
rates_sheet.append(["Запрошенная дата", "Фактическая дата курса", "Курс USD, руб.",
                    "Что вернул ЦБ"])
for requested_date in sorted(rates, key=lambda d: datetime.strptime(d, "%d.%m.%Y")):
    data = rates[requested_date]
    rates_sheet.append([
        requested_date,
        data["actual_date"],
        data["usd_rate"],
        "курс предыдущего рабочего дня" if data["shifted"] else (data["note"] or "курс точно на запрошенную дату"),
    ])
for cell in rates_sheet[1]:
    cell.font = BOLD
for row_index in range(2, rates_sheet.max_row + 1):
    rates_sheet.cell(row=row_index, column=3).number_format = RATE_FORMAT
for column_index, width in enumerate([20, 22, 16, 34], start=1):
    rates_sheet.column_dimensions[get_column_letter(column_index)].width = width
rates_sheet.freeze_panes = "A2"

wb.save(DASHBOARD_FILE)

# --- 4.2. Текстовое резюме -----------------------------------------------------
def as_money(value):
    """198960000 -> '198 960 000'."""
    return "{:,}".format(value).replace(",", " ")


lines = [
    "Юридический скрининг реестра контрагентов — итоги для партнёрского собрания.",
    "Проверено " + str(screen_stats["checked"]) + " контрагентов реестра по чёрному списку Банка России" +
    (", проверка прошла без сбоев." if screen_stats["failed"] == 0
     else ", сбоев при проверке: " + str(screen_stats["failed"]) + "."),
    "В чёрном списке ЦБ найдено " + str(len(blacklisted)) + " контрагентов, из них ликвидировано " +
    str(len(liquidated)) + "; по остальным " + str(screen_stats["not_found"]) +
    " контрагентам признаков нелегальной деятельности не выявлено.",
    "Уровни риска: критический — " + str(level_stats["критический"]["count"]) +
    ", высокий — " + str(level_stats["высокий"]["count"]) +
    ", средний — " + str(level_stats["средний"]["count"]) +
    ", низкий — " + str(level_stats["низкий"]["count"]) + ".",
    "Суммарные обязательства по реестру — " + as_money(total_amount) + " руб., из них на критический риск приходится " +
    as_money(level_stats["критический"]["amount"]) + " руб., на высокий — " +
    as_money(level_stats["высокий"]["amount"]) + " руб.",
    "Рекомендация: " + str(level_stats["критический"]["count"]) +
    " сделок с критическим риском не заключать, " + str(level_stats["высокий"]["count"]) +
    " сделок высокого риска — только после углублённой проверки контрагента.",
    "Платежи по " + str(len(shifted_dates)) + " датам из " + str(len(rates)) +
    " приходятся на дни, когда ЦБ курс не устанавливал, — график платежей требует сверки с календарём.",
]

with open(SUMMARY_FILE, "w", encoding="utf-8") as handle:
    handle.write("\n".join(lines) + "\n")

print("Дашборд сохранён: " + DASHBOARD_FILE + " (листы: " + ", ".join(wb.sheetnames) + ")")
print("Резюме сохранено: " + SUMMARY_FILE + " (" + str(len(lines)) + " строк)")
print()
print("Экспозиция по уровням риска:")
for level in RISK_ORDER:
    print("  " + level.ljust(13) + str(level_stats[level]["count"]).ljust(4) +
          as_money(level_stats[level]["amount"]) + " руб.")
print("  итого       " + str(len(counterparties)).ljust(4) + as_money(total_amount) + " руб.")
print()
print("Текст резюме:")
for line in lines:
    print("  " + line)


Дашборд сохранён: screening_dashboard.xlsx (листы: Сводка, Реестр, Курсы)
Резюме сохранено: screening_summary.txt (7 строк)

Экспозиция по уровням риска:
  критический  8   47 100 000 руб.
  высокий      9   99 350 000 руб.
  средний      8   37 950 000 руб.
  низкий       5   14 560 000 руб.
  итого       30  198 960 000 руб.

Текст резюме:
  Юридический скрининг реестра контрагентов — итоги для партнёрского собрания.
  Проверено 30 контрагентов реестра по чёрному списку Банка России, проверка прошла без сбоев.
  В чёрном списке ЦБ найдено 8 контрагентов, из них ликвидировано 0; по остальным 22 контрагентам признаков нелегальной деятельности не выявлено.
  Уровни риска: критический — 8, высокий — 9, средний — 8, низкий — 5.
  Суммарные обязательства по реестру — 198 960 000 руб., из них на критический риск приходится 47 100 000 руб., на высокий — 99 350 000 руб.
  Рекомендация: 8 сделок с критическим риском не заключать, 9 сделок высокого риска — только после углублённой проверки конт